# 18 · Spatial linkage audit: อุบัติเหตุ → ถนน และคลัง segment ผู้สมัคร (V3, ไม่มีโมเดล)

ตรวจการเชื่อมโยงเหตุ (lineage จาก notebook 16 ที่ตรวจรับแล้ว) กับ snapshot geometry ถนนทางการ 3 หน่วยงาน (DOH/DRR/EXAT จาก notebook 17 v3.1) แบบ **อนุรักษ์นิยม** และสร้างคลัง segment 0.5/1/2 กม. จาก geometry ถนน (รวมถนนที่ไม่มีเหตุ)

- อ่านเฉพาะ input ที่ hash ตรงกับ config และ manifest ที่ตรวจรับแล้ว (offline เท่านั้น) ไม่ตรง = หยุดทันที ไม่มี acceptance manifest
- ระยะทางคำนวณใน WGS84 / UTM 47N (EPSG:32647, always_xy) โดย **สมมติ** datum ของพิกัดเหตุเป็น WGS84 (ยังไม่ยืนยัน) ไม่แปลง degree เป็นเมตร
- จับคู่ต้อง agency ตรงตัว + รหัสสายทางที่ normalize ตามกฎและยืนยันว่ามีใน snapshot + ระยะ ≤ 50 ม. (sensitivity 25/100 ม.) มีคู่แข่งภายใน 5 ม. หรือ geometry ไม่ปลอดภัย = ambiguous; EXAT ไม่มี crosswalk = identity_unresolved
- ไม่ซ่อม geometry/พิกัด ไม่ใช้ KM เป็นเกณฑ์ ไม่ train ไม่สร้าง panel ที่เติมศูนย์ ไม่มี probability/threshold/สีความเสี่ยง; ไม่มีบันทึก ≠ ไม่มีอุบัติเหตุ

**Runbook (VS Code):** เลือก kernel `.venv` แล้ว Run All (เขียนได้เฉพาะ dir ใหม่ `backend/reports/v3/linkage_audit` และ `backend/data/v3/linkage_audit`) หรือให้ Codex รันแบบแยก root:

`.venv/bin/python backend/scripts/run_spatial_linkage_audit.py --output-dir backend/rerun-linkage-014-001`

ทุกครั้งใช้ output root ใหม่ ห้ามเขียนทับผลที่ตรวจรับแล้ว

In [ ]:
# PARAMETERS (explicit; the runner sets them through environment variables, VS Code users may edit them here)
import os
from pathlib import Path


def _default_backend(c):
    for cand in (c, c.parent, c / 'backend'):
        if (cand / 'config' / 'spatial_linkage_v3.json').is_file():
            return cand
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_PATH = BACKEND / 'config' / 'spatial_linkage_v3.json'
print('backend:', BACKEND, '| output root:', OUTPUT_ROOT)

In [ ]:
# FUNCTIONS (1/4): errors, hashing, atomic writes, output guards, config validation and fail-closed input verification. Pure logic; the tests exec these cells.
import datetime as dt
import gzip
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

NL = chr(10)


class LinkageError(RuntimeError):
    pass


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def atomic_write_bytes(path, data):
    # sibling temp file + rename: a reader never sees a half-written file
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'wb') as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def jsonable(o):
    # numpy scalars to Python, non-finite floats to None (allow_nan=False keeps the JSON strict)
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, Path):
        return str(o)
    if o is pd.NA:
        return None
    return o


def atomic_write_json(path, obj):
    atomic_write_bytes(path, json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False).encode('utf-8'))


def csv_bytes(df, gz=False):
    b = df.to_csv(index=False).encode('utf-8')
    return gzip.compress(b, mtime=0) if gz else b


def module_version(name):
    if importlib.util.find_spec(name) is None:
        return None
    return str(getattr(importlib.import_module(name), '__version__', 'unknown'))


def require_modules(names, pins):
    # clear failure with the pin to propose to Codex; nothing is installed here
    missing = [n for n in names if importlib.util.find_spec(n) is None]
    if missing:
        raise LinkageError('missing Python package(s) ' + ', '.join(missing) + '; ask Codex to install ' +
                           ' '.join(pins.get(n, n) for n in missing) + '. Nothing was written.')
    return True


def as_flag(values):
    # notebook 16 writes booleans as text; only an explicit true counts (missing is not true)
    return pd.Series(list(values), dtype=object).astype(str).str.strip().str.lower().isin(['true', '1', 'yes']).to_numpy()


def tol_label(t):
    t = float(t)
    return (str(int(t)) if t.is_integer() else str(t)) + 'm'


def tolerances(cfg):
    m = cfg['matching']
    return sorted(set([float(m['primary_tolerance_m'])] + [float(t) for t in m['sensitivity_tolerances_m']]))


def validate_config(cfg):
    env = cfg['envelope_epsg4326']
    if len(env) != 4 or not (-180 <= env[0] < env[2] <= 180 and -90 <= env[1] < env[3] <= 90):
        raise LinkageError('invalid envelope ' + str(env))
    c = cfg['crs']
    if c['geographic_epsg'] != 4326 or c['metric_epsg'] != 32647 or c['always_xy'] is not True:
        raise LinkageError('CRS must be EPSG:4326 -> EPSG:32647 (WGS84 / UTM 47N) with always_xy=True')
    m, tols = cfg['matching'], tolerances(cfg)
    if len(tols) != 1 + len(m['sensitivity_tolerances_m']) or tols[0] <= 0 or m['ambiguity_margin_m'] < 0:
        raise LinkageError('tolerances must be distinct positive metres and the ambiguity margin non-negative')
    agencies = [s['agency_exact'] for s in cfg['sources'].values()]
    if len(set(agencies)) != len(agencies):
        raise LinkageError('duplicate agency strings in the exact crosswalk')
    for sid, s in cfg['sources'].items():
        if s['normalizer'] not in NORMALIZERS:
            raise LinkageError(sid + ': unknown normalizer ' + str(s['normalizer']))
    for sid, amap in (cfg.get('alias_map') or {}).items():
        if sid not in cfg['sources']:
            raise LinkageError('alias_map source ' + sid + ' is not configured')
        for k, v in amap.items():
            if not isinstance(v, dict) or not v.get('target') or not v.get('evidence'):
                raise LinkageError('alias ' + sid + ':' + str(k) + ' has no target/evidence; unverified aliases are refused')
    res = cfg['segmentation']['resolutions_m']
    if sorted(set(res)) != list(res) or min(res) <= 0:
        raise LinkageError('resolutions must be increasing positive metres')
    y = cfg['years']
    if set(y) != {'train', 'seen_diagnostic'} or set(y['train']) & set(y['seen_diagnostic']):
        raise LinkageError('years must define disjoint train and seen_diagnostic')
    return tols


def output_paths(out_root, cfg):
    out_root = Path(out_root)
    rep, data = out_root / cfg['outputs']['report_dir'], out_root / cfg['outputs']['data_dir']
    return {'out_root': out_root, 'report_dir': rep, 'figure_dir': rep / 'figures', 'data_dir': data}


def _same(a, b):
    try:
        return os.path.samefile(a, b)
    except OSError:
        return False


def _inside(child, parent):
    # child is parent or beneath it: by resolved path (symlinks, ..) or by file identity of an existing ancestor (aliases)
    c, p = Path(child).resolve(), Path(parent).resolve()
    if c == p or p in c.parents:
        return True
    return p.exists() and any(_same(a, p) for a in (c, *c.parents) if a.exists())


def guarded_paths(backend, cfg):
    backend = Path(backend)
    out = [backend / rel for rel in cfg['protected_paths']]
    out += [backend / v['path'] for v in cfg['inputs'].values() if isinstance(v, dict) and 'path' in v]
    return out + [backend / cfg['inputs']['road_derived_dir']]


def check_outputs(paths, backend, cfg):
    # call before any write: no overlap (either direction, symlinks resolved) with protected or read inputs; both dirs must be fresh
    hits = []
    for label in ('report_dir', 'data_dir'):
        for p in guarded_paths(backend, cfg):
            if _inside(paths[label], p) or _inside(p, paths[label]):
                hits.append(label + ' ' + str(paths[label]) + ' overlaps protected ' + str(p))
    if _inside(paths['report_dir'], paths['data_dir']) or _inside(paths['data_dir'], paths['report_dir']):
        hits.append('report and data dirs overlap')
    if hits:
        raise LinkageError('refusing output root (nothing written): ' + '; '.join(hits))
    used = [str(paths[k]) for k in ('report_dir', 'data_dir') if paths[k].exists() and (not paths[k].is_dir() or any(paths[k].iterdir()))]
    if used:
        raise LinkageError('earlier output exists in ' + ', '.join(used) + '; use a NEW output root, e.g. backend/rerun-linkage-<id>')
    return True


def check_report_file(rel, cfg):
    # the tracked report dir receives aggregated evidence only (no GeoJSON / gz / row-level files)
    if Path(rel).suffix.lower() not in cfg['report_allowed_suffixes']:
        raise LinkageError('refusing to write ' + str(rel) + ' into the report dir (aggregated csv/png/md/json only)')
    return True


def hash_paths(root, rels):
    # relative path -> sha256 for every file under each protected rel; a missing rel is recorded as None
    root, out = Path(root), {}
    for rel in rels:
        p = root / rel
        if p.is_file():
            out[Path(rel).as_posix()] = sha256_file(p)
        elif p.is_dir():
            for q in sorted(p.rglob('*')):
                if q.is_file():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = sha256_file(q)
        else:
            out[Path(rel).as_posix()] = None
    return out


def verify_pinned(path, expected, label):
    path = Path(path)
    if not expected:
        raise LinkageError(label + ': sha256 is not pinned in the config (fail-closed). Pin the accepted value, e.g. shasum -a 256 ' + str(path))
    if not path.is_file():
        raise LinkageError(label + ': missing input ' + str(path))
    got = sha256_file(path)
    if got != expected:
        raise LinkageError(label + ': sha256 mismatch for ' + str(path) + ' (expected ' + expected + ', got ' + got + ')')
    return got


def verify_inputs(backend, cfg):
    # every hash is checked before analysis; returns (verification rows, accepted documents, manifest-derived expectations)
    backend, inp, rows, docs = Path(backend), cfg['inputs'], [], {}
    for key in ('road_manifest', 'road_review', 'readiness_manifest'):
        spec = inp[key]
        got = verify_pinned(backend / spec['path'], spec['sha256'], key)
        rows.append({'input': key, 'path': spec['path'], 'sha256': got, 'rule': 'config pin', 'status': 'pass'})
        docs[key] = json.loads((backend / spec['path']).read_text('utf-8'))
        if docs[key].get('status') != spec['expect_status']:
            raise LinkageError(key + ': status ' + repr(docs[key].get('status')) + ' is not the accepted ' + repr(spec['expect_status']))
    man = docs['road_manifest']
    if man.get('version') != inp['road_manifest']['expect_version'] or man.get('cache_manifest_sha256') != inp['road_manifest']['expect_cache_manifest_sha256']:
        raise LinkageError('road manifest version/cache identity differs from the accepted v3.1 snapshot')
    if [float(v) for v in man['preregistration']['envelope_epsg4326']] != [float(v) for v in cfg['envelope_epsg4326']]:
        raise LinkageError('config envelope differs from the road manifest preregistration')
    if cfg['snapshot']['tag'] != 'rc31' + man['cache_manifest_sha256'][:8]:
        raise LinkageError('snapshot tag does not follow the registered rule rc31 + cache_manifest_sha256[:8]')
    derived = man.get('derived_files') or {}
    if set(derived) != set(inp['road_derived_sha256']):
        raise LinkageError('derived file set of the road manifest differs from the config pins')
    for name in sorted(derived):
        if derived[name] != inp['road_derived_sha256'][name]:
            raise LinkageError('config pin for ' + name + ' differs from the road manifest')
        got = verify_pinned(backend / inp['road_derived_dir'] / name, derived[name], name)
        rows.append({'input': name, 'path': inp['road_derived_dir'] + '/' + name, 'sha256': got, 'rule': 'config pin == road manifest derived_files', 'status': 'pass'})
    lin = inp['lineage']
    entry = [e for e in docs['readiness_manifest'].get('row_level_outputs_gitignored', []) if e.get('path') == lin['manifest_key']]
    if len(entry) != 1 or entry[0].get('sha256') != lin['sha256']:
        raise LinkageError('lineage pin is not the readiness manifest entry ' + lin['manifest_key'])
    got = verify_pinned(backend / lin['path'], lin['sha256'], 'lineage')
    rows.append({'input': 'lineage', 'path': lin['path'], 'sha256': got, 'rule': 'config pin == readiness manifest row_level_outputs_gitignored', 'status': 'pass'})
    expect = {'lineage_rows': int(entry[0]['rows']), 'retained_events_linked': int(docs['readiness_manifest']['linkage']['retained_events_linked']),
              'road_counts': {sid: int(v['count']) for sid, v in man['sources'].items()}}
    if expect['lineage_rows'] != expect['retained_events_linked']:
        raise LinkageError('readiness manifest lineage rows differ from retained_events_linked')
    if set(expect['road_counts']) != set(cfg['sources']):
        raise LinkageError('road manifest sources differ from the configured sources')
    for sid, n in expect['road_counts'].items():
        if int(docs['road_review']['counts'][sid]['exact_ids']) != n:
            raise LinkageError(sid + ': independent review exact_ids differ from the road manifest count')
    return pd.DataFrame(rows), docs, expect

In [ ]:
# FUNCTIONS (2/4): event lineage, coordinate flags (never repaired), exact agency crosswalk and per-agency route normalization.
LINEAGE_REQUIRED = ['event_uid', 'source_file', 'source_sha256', 'source_row', 'ce_year', 'event_date', 'time_block', 'province_key', 'split',
                    'raw_latitude', 'raw_longitude', 'raw_route_code', 'raw_route_name', 'raw_km', 'raw_agency', 'raw_road_agency_type',
                    'lat_value', 'lon_value', 'coord_status', 'lat_decimals', 'lon_decimals', 'low_precision_flag', 'lat_equals_lon_flag',
                    'n_events_same_coord', 'repeated_coord_flag', 'km_value', 'km_status']
THAI_CONSONANTS = frozenset('กขฃคฅฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ')
ASCII_DIGITS = frozenset('0123456789')
IDENTITY_STATUSES = ('agency_unmapped', 'route_missing', 'route_nonstandard', 'route_special_unverified', 'identity_unresolved',
                     'route_not_in_snapshot', 'verified')


def check_events(df, expect_rows, cfg):
    uid = cfg['event_fields']['uid']
    if len(df) != expect_rows:
        raise LinkageError('lineage has ' + str(len(df)) + ' rows but the accepted readiness manifest records ' + str(expect_rows))
    if (df[uid].astype(str).str.strip() == '').any():
        raise LinkageError('empty event_uid in lineage')
    dup = df[uid].duplicated(keep=False)
    if dup.any():
        ex = sorted(df.loc[dup, uid].astype(str).unique())[:5]
        raise LinkageError('duplicate event_uid in lineage: ' + str(int(dup.sum())) + ' rows (e.g. ' + ', '.join(ex) + '); no output was produced')
    years = {int(y): k for k, v in cfg['years'].items() for y in v}
    mapped = pd.to_numeric(df['ce_year'], errors='coerce').map(years)
    bad = mapped.isna().to_numpy() | (mapped.astype(object).to_numpy() != df['split'].to_numpy(object))
    if bad.any():
        raise LinkageError(str(int(bad.sum())) + ' lineage rows have a split that disagrees with the configured years')
    return df


def load_lineage(path, expect_rows, cfg):
    # every column read as text so leading zeros (route codes, decimals) survive
    df = pd.read_csv(path, dtype=str, keep_default_na=False, compression='gzip', encoding='utf-8')
    missing = [c for c in LINEAGE_REQUIRED if c not in df.columns]
    if missing:
        raise LinkageError('lineage is missing columns ' + ', '.join(missing))
    return check_events(df, expect_rows, cfg).reset_index(drop=True)


def coordinate_audit(df, cfg):
    # values parsed as given (no swap/rounding repair). Usable = notebook 16 usable class AND finite; inside-envelope is a scope filter only
    lat = pd.to_numeric(df['lat_value'], errors='coerce').to_numpy(float)
    lon = pd.to_numeric(df['lon_value'], errors='coerce').to_numpy(float)
    claimed = df['coord_status'].to_numpy(object) == cfg['event_fields']['usable_coord_status']
    finite = np.isfinite(lat) & np.isfinite(lon)
    if (claimed & ~finite).any():
        raise LinkageError(str(int((claimed & ~finite).sum())) + ' rows have a usable coord_status but non-finite values')
    usable = claimed & finite
    e = cfg['envelope_epsg4326']
    with np.errstate(invalid='ignore'):
        inside = usable & (lon >= e[0]) & (lon <= e[2]) & (lat >= e[1]) & (lat <= e[3])
    lowp, eq, rep = as_flag(df['low_precision_flag']), as_flag(df['lat_equals_lon_flag']), as_flag(df['repeated_coord_flag'])
    scope = np.where(inside, 'pilot', np.where(usable, 'outside_envelope', 'coord_not_usable'))
    prec = np.where(~usable, 'not_usable', np.where(lowp | eq, 'low_precision_unsuitable', 'adequate_unverified'))
    return pd.DataFrame({'lat': lat, 'lon': lon, 'coord_usable': usable, 'scope': scope, 'precision_class': prec, 'low_precision': lowp,
                         'lat_equals_lon': eq, 'repeated_coord': rep}, index=df.index)


def _nfkc(s):
    return unicodedata.normalize('NFKC', '' if s is None else str(s))


def norm_doh(raw):
    # strict ASCII digits (1-4) then zfill(4): '1' -> '0001'; anything else stays unverified
    s = _nfkc(raw).strip()
    if s == '':
        return 'route_missing', None
    if 1 <= len(s) <= 4 and all(ch in ASCII_DIGITS for ch in s):
        return 'standard', s.zfill(4)
    return 'route_nonstandard', None


def norm_drr(raw):
    # whitespace removed, repeated dots collapsed, edge dots stripped; Thai prefix and leading zeros kept; no padding or inserted dot
    s = ''.join(_nfkc(raw).split())
    while '..' in s:
        s = s.replace('..', '.')
    s = s.strip('.')
    if s == '':
        return 'route_missing', None
    head, dot, tail = s.partition('.')
    digits = tail != '' and all(ch in ASCII_DIGITS for ch in tail)
    if dot and len(head) == 2 and all(ch in THAI_CONSONANTS for ch in head) and len(tail) == 4 and digits:
        return 'standard', s
    if dot and head == 'ส' and digits:
        return 'route_special_unverified', s
    return 'route_nonstandard', None


def norm_exat(raw):
    # no verified crosswalk from accident route codes to EXAT geometry routes
    return ('route_missing', None) if _nfkc(raw).strip() == '' else ('identity_unresolved', None)


NORMALIZERS = {'doh_numeric_zfill4': norm_doh, 'drr_prefix_digits': norm_drr, 'exat_unresolved': norm_exat}


def normalize_route(sid, raw, cfg):
    return NORMALIZERS[cfg['sources'][sid]['normalizer']](raw)


def event_identity(df, cfg, route_idx):
    # exact agency string (NFKC + edge strip only, no substring/fuzzy), per-agency normalization, alias map only by exact key with evidence
    f = cfg['event_fields']
    agency_to_sid = {_nfkc(s['agency_exact']).strip(): sid for sid, s in cfg['sources'].items()}
    rows = []
    for ag, rc in zip(df[f['agency']].tolist(), df[f['route']].tolist()):
        sid = agency_to_sid.get(_nfkc(ag).strip())
        if sid is None:
            rows.append((None, 'agency_unmapped', None, 'agency_not_in_exact_crosswalk', None))
            continue
        spec = cfg['sources'][sid]
        st, norm = NORMALIZERS[spec['normalizer']](rc)
        rule, key = spec['normalizer'], norm
        alias = (cfg.get('alias_map') or {}).get(sid, {}).get(norm) if norm is not None else None
        if alias:
            rule, key, st = rule + '+alias', alias['target'], 'standard'
        if st == 'standard':
            st = 'verified' if (sid, key) in route_idx else 'route_not_in_snapshot'
        rows.append((sid, st, norm, rule, key if st == 'verified' else None))
    return pd.DataFrame(rows, columns=['source_id', 'identity_status', 'route_normalized', 'normalization_rule', 'route_key_matched'], index=df.index)

In [ ]:
# FUNCTIONS (3/4): metric projection, road features/parts/clipped pieces, candidate search, conservative status rules and segmentation.
PRE_STATUSES = ('coord_not_usable', 'outside_pilot_envelope', 'low_precision_unsuitable')
LINK_STATUSES = ('too_far', 'ambiguous_unsafe_geometry', 'ambiguous_multiple', 'matched')
STATUSES = PRE_STATUSES + IDENTITY_STATUSES[:-1] + LINK_STATUSES
_GEOD = []


def geod():
    if not _GEOD:
        from pyproj import Geod
        _GEOD.append(Geod(ellps='WGS84'))
    return _GEOD[0]


def geod_length(coords):
    a = np.asarray(coords, float)
    return float(geod().line_length(a[:, 0], a[:, 1])) if len(a) >= 2 else 0.0


def make_transformers(cfg):
    from pyproj import Transformer
    c = cfg['crs']
    g, m = 'EPSG:' + str(c['geographic_epsg']), 'EPSG:' + str(c['metric_epsg'])
    return Transformer.from_crs(g, m, always_xy=c['always_xy']), Transformer.from_crs(m, g, always_xy=c['always_xy'])


def project_xy(tr, coords):
    a = np.asarray(coords, float)
    if len(a) == 0:
        return np.zeros((0, 2))
    x, y = tr.transform(a[:, 0], a[:, 1])
    return np.column_stack([np.asarray(x, float), np.asarray(y, float)])


def line_parts(g):
    # line pieces of a shapely result (clip/intersection may return lines, collections or points); points carry no length
    if g is None or g.is_empty:
        return []
    if g.geom_type in ('LineString', 'LinearRing'):
        return [list(g.coords)] if len(g.coords) >= 2 else []
    if g.geom_type in ('MultiLineString', 'GeometryCollection'):
        out = []
        for h in g.geoms:
            out += line_parts(h)
        return out
    return []


def geojson_line_parts(gj):
    if gj['type'] == 'LineString':
        return [gj['coordinates']]
    if gj['type'] == 'MultiLineString':
        return list(gj['coordinates'])
    raise LinkageError('unexpected geometry type ' + str(gj.get('type')))


def check_geometry_ids(sid, geo_ids, diag_rows, expected, oid_values=None):
    # source geometry IDs must reconcile exactly with the accepted manifest count
    oids = [int(o) for o in diag_rows['oid']]
    if expected is None or len(oids) != int(expected):
        raise LinkageError(sid + ': ' + str(len(oids)) + ' diagnostics rows but the road manifest count is ' + str(expected))
    if len(set(oids)) != len(oids):
        raise LinkageError(sid + ': duplicate OID in feature_diagnostics')
    ids = [int(i) for i in geo_ids]
    if len(set(ids)) != len(ids):
        raise LinkageError(sid + ': duplicate geometry id in the full GeoJSON')
    ok = set(int(o) for o, g in zip(diag_rows['oid'], as_flag(diag_rows['geojson_ok'])) if g)
    if set(ids) != ok:
        raise LinkageError(sid + ': full GeoJSON ids differ from the diagnostics geojson_ok OIDs')
    if oid_values is not None and [None if v is None else int(v) for v in oid_values] != ids:
        raise LinkageError(sid + ': a GeoJSON feature id differs from its OID attribute')
    return True


def load_features(derived_dir, cfg, expect_counts):
    # full official sections exactly as accepted (lineage) + diagnostics; any count/ID mismatch fails
    derived_dir = Path(derived_dir)
    diag = pd.read_csv(derived_dir / 'feature_diagnostics.csv', dtype=str, keep_default_na=False)
    extra = sorted(set(diag['source_id']) - set(cfg['sources']))
    if extra:
        raise LinkageError('diagnostics contain unconfigured sources ' + ', '.join(extra))
    feats = []
    for sid, spec in cfg['sources'].items():
        gj = json.loads((derived_dir / (sid + '_full.geojson')).read_text('utf-8'))
        fs = gj.get('features') or []
        props = [ft.get('properties') or {} for ft in fs]
        check_geometry_ids(sid, [ft.get('id') for ft in fs], diag[diag['source_id'] == sid], expect_counts.get(sid), [p.get(spec['oid_field']) for p in props])
        for ft, p in zip(fs, props):
            feats.append({'source_id': sid, 'oid': int(ft['id']), 'geometry': ft['geometry'], 'route_raw': p.get(spec['route_field']), 'props': p})
    return feats, diag


def clipped_ids(derived_dir, cfg):
    out = {}
    for sid in cfg['sources']:
        gj = json.loads((Path(derived_dir) / (sid + '_clipped.geojson')).read_text('utf-8'))
        out[sid] = sorted(int(ft['id']) for ft in gj.get('features') or [])
    return out


def on_envelope_edge(pt, env, tol=1e-9):
    x, y = float(pt[0]), float(pt[1])
    return abs(x - env[0]) <= tol or abs(x - env[2]) <= tol or abs(y - env[1]) <= tol or abs(y - env[3]) <= tol


PART_COLS = ['part_uid', 'source_id', 'oid', 'part', 'n_parts', 'route_raw', 'route_status', 'route_key', 'geom_utm', 'len_utm_m', 'len_geod_m']
PIECE_COLS = ['part_uid', 'source_id', 'oid', 'part', 'piece', 'route_raw', 'route_status', 'route_key', 'geom_utm', 'coords_ll', 'len_utm_m',
              'len_geod_m', 'start_on_part_m', 'end_on_part_m', 'clipped_at_start', 'clipped_at_end']


def build_roads(feats, diag, cfg, fwd):
    # parts = full-section parts (lineage); pieces = each full part clipped to the envelope in EPSG:4326, ordered along the part.
    # Unsafe features (not simple/valid, short or zero-length parts, swapped/non-finite) stay in the audit with reasons; nothing is repaired.
    import shapely
    from shapely.geometry import LineString, Point, shape
    env = cfg['envelope_epsg4326']
    dmap = {(r['source_id'], int(r['oid'])): r for r in diag.to_dict('records')}
    feat_rows, parts, pieces = [], [], []
    for f in feats:
        sid, oid = f['source_id'], f['oid']
        d = dmap[(sid, oid)]
        g = shape(f['geometry'])
        valid, simple = bool(g.is_valid), bool(g.is_simple)
        if valid != bool(as_flag([d['shapely_valid']])[0]) or simple != bool(as_flag([d['shapely_simple']])[0]):
            raise LinkageError(sid + ' ' + str(oid) + ': validity/simplicity differs from the accepted diagnostics')
        reasons = ([] if valid else ['not_valid']) + ([] if simple else ['not_simple'])
        reasons += [k for k in ('empty_or_short_part', 'nonfinite_coord', 'curve_geometry', 'latlon_swap_suspect') if as_flag([d.get(k, '')])[0]]
        st, key = normalize_route(sid, f['route_raw'], cfg)
        lp = geojson_line_parts(f['geometry'])
        fparts, fpieces, full_geod, clip_geod = [], [], 0.0, 0.0
        for k, pc in enumerate(lp):
            ll = np.asarray(pc, float)[:, :2]
            utm = LineString(project_xy(fwd, ll))
            if not utm.length > 0:
                reasons.append('zero_length_part')
            pg = geod_length(ll)
            full_geod += pg
            puid = len(parts) + len(fparts)
            fparts.append({'part_uid': puid, 'source_id': sid, 'oid': oid, 'part': k, 'n_parts': len(lp), 'route_raw': f['route_raw'], 'route_status': st,
                           'route_key': key if st == 'standard' else None, 'geom_utm': utm, 'len_utm_m': utm.length, 'len_geod_m': pg})
            tmp = []
            for cc in line_parts(shapely.clip_by_rect(LineString(ll), *env)):
                cll = np.asarray(cc, float)[:, :2]
                cu = LineString(project_xy(fwd, cll))
                cg = geod_length(cll)
                clip_geod += cg
                tmp.append({'part_uid': puid, 'source_id': sid, 'oid': oid, 'part': k, 'route_raw': f['route_raw'], 'route_status': st,
                            'route_key': key if st == 'standard' else None, 'geom_utm': cu, 'coords_ll': cll, 'len_utm_m': cu.length, 'len_geod_m': cg,
                            'start_on_part_m': utm.project(Point(cu.coords[0])), 'end_on_part_m': utm.project(Point(cu.coords[-1])),
                            'clipped_at_start': on_envelope_edge(cll[0], env), 'clipped_at_end': on_envelope_edge(cll[-1], env)})
            tmp.sort(key=lambda r: (r['start_on_part_m'], r['end_on_part_m']))
            for i, r in enumerate(tmp):
                r['piece'] = i
            fpieces += tmp
        why = ';'.join(sorted(set(reasons)))
        for r in fparts + fpieces:
            r['unsafe'], r['unsafe_reasons'] = bool(reasons), why
        parts += fparts
        pieces += fpieces
        dl = d.get('length_clipped_m', '')
        feat_rows.append({'source_id': sid, 'oid': oid, 'n_parts': len(lp), 'route_raw': f['route_raw'], 'route_status': st, 'route_key': key,
                          'valid': valid, 'simple': simple, 'unsafe': bool(reasons), 'unsafe_reasons': why, 'n_pieces': len(fpieces),
                          'len_full_geod_m': full_geod, 'len_clipped_geod_m': clip_geod, 'diag_length_full_m': float(d['length_full_m']),
                          'diag_length_clipped_m': float(dl) if dl not in ('', None) else 0.0})
    P = pd.DataFrame(parts, columns=PART_COLS + ['unsafe', 'unsafe_reasons'])
    C = pd.DataFrame(pieces, columns=PIECE_COLS + ['unsafe', 'unsafe_reasons'])
    C.insert(0, 'piece_uid', np.arange(len(C), dtype=int))
    for df in (P, C):
        df['unsafe'] = df['unsafe'].astype(bool)
    if not (P['part_uid'].to_numpy() == np.arange(len(P))).all():
        raise LinkageError('internal: part_uid must equal the row position')
    return pd.DataFrame(feat_rows), P, C


def route_index(parts):
    # (source, normalized route) -> part_uids; only standard identities are indexed, special/nonstandard codes never match
    idx = {}
    for r in parts[parts['route_key'].notna()].itertuples(index=False):
        idx.setdefault((r.source_id, r.route_key), []).append(int(r.part_uid))
    return idx


def piece_overlaps(pieces, min_len):
    # shared length between clipped pieces (any source pair); flagged only, never merged or deduplicated
    import shapely
    from shapely.geometry import LineString
    from shapely.strtree import STRtree
    n = len(pieces)
    flags = pd.DataFrame({'overlap_m': np.zeros(n), 'overlap_flag': np.zeros(n, bool), 'duplicate_flag': np.zeros(n, bool)}, index=pieces.index)
    rows, failed = [], 0
    if n < 2:
        return pd.DataFrame(rows, columns=['source_a', 'oid_a', 'source_b', 'oid_b', 'overlap_m', 'exact_duplicate']), flags, failed
    geoms = np.asarray(pieces['geom_utm'].tolist(), dtype=object)
    a, b = STRtree(geoms).query(geoms, predicate='intersects')
    keep = a < b
    src, oid = pieces['source_id'].to_numpy(object), pieces['oid'].to_numpy()
    for i, j in zip(a[keep], b[keep]):
        try:
            L = sum(LineString(c).length for c in line_parts(geoms[i].intersection(geoms[j])))
        except Exception:
            failed += 1
            continue
        if L >= min_len:
            dup = bool(geoms[i].equals(geoms[j]))
            rows.append({'source_a': src[i], 'oid_a': int(oid[i]), 'source_b': src[j], 'oid_b': int(oid[j]), 'overlap_m': L, 'exact_duplicate': dup})
            for k in (i, j):
                flags.iloc[k, 0] += L
                flags.iloc[k, 1] = True
                flags.iloc[k, 2] = flags.iloc[k, 2] or dup
    return pd.DataFrame(rows, columns=['source_a', 'oid_a', 'source_b', 'oid_b', 'overlap_m', 'exact_duplicate']), flags, failed


def candidate_pairs(points_xy, part_geoms, radius):
    # metric STRtree over road parts: radius-box query then exact point-line distances (never events x all vertices)
    import shapely
    from shapely.strtree import STRtree
    geoms = np.asarray(part_geoms, dtype=object)
    empty = pd.DataFrame({'ev': np.array([], int), 'part_uid': np.array([], int), 'dist_m': np.array([], float)})
    if len(points_xy) == 0 or len(geoms) == 0:
        return empty
    x, y = points_xy[:, 0], points_xy[:, 1]
    ei, pi = STRtree(geoms).query(shapely.box(x - radius, y - radius, x + radius, y + radius))
    if len(ei) == 0:
        return empty
    d = shapely.distance(shapely.points(x[ei], y[ei]), geoms[pi])
    k = d <= radius
    return pd.DataFrame({'ev': ei[k].astype(int), 'part_uid': pi[k].astype(int), 'dist_m': d[k]})


def nearest_within(points_xy, geoms, max_d):
    # diagnostic only: distance to the nearest geometry within max_d (NaN beyond); never used for acceptance
    import shapely
    from shapely.strtree import STRtree
    out_d, out_i = np.full(len(points_xy), np.nan), np.full(len(points_xy), -1)
    if len(points_xy) == 0 or len(geoms) == 0:
        return out_d, out_i
    tree = STRtree(np.asarray(geoms, dtype=object))
    (ei, gi), d = tree.query_nearest(shapely.points(points_xy[:, 0], points_xy[:, 1]), max_distance=max_d, return_distance=True, all_matches=False)
    out_d[ei], out_i[ei] = d, gi
    return out_d, out_i


def pre_status(ev):
    s = np.where(ev['scope'] == 'coord_not_usable', 'coord_not_usable',
                 np.where(ev['scope'] == 'outside_envelope', 'outside_pilot_envelope',
                          np.where(ev['precision_class'] == 'low_precision_unsuitable', 'low_precision_unsuitable', ev['identity_status'].astype(str))))
    return s.astype(object)


def assign_status(ev, pairs, parts, tols, margin):
    # ev: RangeIndex rows with scope, precision_class, source_id, identity_status, route_key_matched; pairs: (ev, part_uid, dist_m) within radius.
    # Eligible = same source AND same verified route key; contenders = eligible within best+margin (also beyond the tolerance) -> statuses nest.
    n, ix = len(ev), np.arange(len(ev))
    pre = pre_status(ev)
    p = pairs.merge(parts[['part_uid', 'source_id', 'route_key', 'oid', 'part', 'unsafe']], on='part_uid', how='left')
    evi = p['ev'].to_numpy(int)
    p['ev_source'] = ev['source_id'].to_numpy(object)[evi]
    p['ev_route'] = ev['route_key_matched'].to_numpy(object)[evi]
    same = (p['ev_route'].notna() & (p['source_id'] == p['ev_source']) & (p['route_key'] == p['ev_route'])).to_numpy(bool)
    el = p[same].sort_values(['ev', 'dist_m', 'source_id', 'oid', 'part'], kind='mergesort').reset_index(drop=True)
    el['rank'] = el.groupby('ev').cumcount()
    best = el[el['rank'] == 0].set_index('ev')
    second = el[el['rank'] == 1].set_index('ev')['dist_m']
    el['best'] = best['dist_m'].reindex(el['ev']).to_numpy()
    cont = el[el['dist_m'] <= el['best'] + margin]
    out = pd.DataFrame(index=ev.index)
    bd = best['dist_m'].reindex(ix).to_numpy(float)
    nc = cont.groupby('ev').size().reindex(ix).fillna(0).astype(int).to_numpy()
    uc = cont.groupby('ev')['unsafe'].any().reindex(ix).fillna(False).astype(bool).to_numpy()
    out['best_same_identity_m'] = bd
    out['second_same_identity_m'] = second.reindex(ix).to_numpy(float)
    out['n_contenders_within_margin'] = nc
    out['contender_unsafe'] = uc
    other = p[~same].copy()
    other['best'] = best['dist_m'].reindex(other['ev']).to_numpy()
    # other-agency/other-route geometry inside best+margin is a flag only: the recorded identity decides eligibility
    out['cross_identity_within_margin'] = np.isin(ix, other.loc[other['dist_m'] <= other['best'] + margin, 'ev'].unique())
    verified = pre == 'verified'
    with np.errstate(invalid='ignore'):
        for t in tols:
            lab = tol_label(t)
            out['n_eligible_' + lab] = el[el['dist_m'] <= t].groupby('ev').size().reindex(ix).fillna(0).astype(int).to_numpy()
            hit = verified & (bd <= t)
            st = pre.copy()
            st[verified & ~hit] = 'too_far'
            st[hit & uc] = 'ambiguous_unsafe_geometry'
            st[hit & ~uc & (nc > 1)] = 'ambiguous_multiple'
            st[hit & ~uc & (nc == 1)] = 'matched'
            out['status_' + lab] = st
    chosen = best[['oid', 'part', 'part_uid', 'dist_m']].reindex(ix)
    return out, chosen


def n_segments(length, R, eps):
    # a remainder shorter than eps (float noise) is absorbed by the previous segment instead of creating a sliver
    return max(1, int(math.ceil((length - eps) / R)))


def segment_index(t, length, R, eps):
    # half-open [i*R, (i+1)*R) along the digitised direction; the last segment is closed at the piece end
    if not (t >= 0 and t <= length + eps):
        raise LinkageError('offset ' + str(t) + ' outside the piece length ' + str(length))
    return min(int(math.floor(t / R)), n_segments(length, R, eps) - 1)


def segment_id(version, tag, short, oid, part, piece, R, i):
    return version + '-' + tag + ':' + short + ':' + str(int(oid)) + ':p' + str(int(part)) + ':c' + str(int(piece)) + ':r' + str(int(R)) + ':' + format(int(i), '05d')


def cut_line(xy, cuts):
    # metric polyline cut at increasing along-line distances [0, c1, ..., L]; vertices kept, cut points interpolated linearly
    xy = np.asarray(xy, float)
    seg = np.hypot(np.diff(xy[:, 0]), np.diff(xy[:, 1]))
    cum = np.concatenate([[0.0], np.cumsum(seg)])

    def at(dd):
        j = min(max(int(np.searchsorted(cum, dd, side='right')) - 1, 0), len(seg) - 1)
        r = 0.0 if seg[j] == 0 else min(max((dd - cum[j]) / seg[j], 0.0), 1.0)
        return xy[j] + r * (xy[j + 1] - xy[j])
    out = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        inner = xy[(cum > a) & (cum < b)]
        out.append(np.vstack([at(a), inner, at(b)]))
    return out


SEG_COLS = ['segment_id', 'resolution_m', 'snapshot_tag', 'source_id', 'oid', 'part', 'piece', 'piece_uid', 'seg_index', 'n_in_piece', 'start_m', 'end_m',
            'length_m', 'length_geod_m', 'short_final', 'clipped_at_start', 'clipped_at_end', 'piece_overlap', 'piece_duplicate', 'route_raw',
            'route_key', 'route_status', 'coords_utm', 'coords_ll']


def build_segments(pieces, cfg, inv):
    # universe from GEOMETRY only (roads without recorded crashes included); unsafe features are excluded (reasons kept in the audit)
    sc, tag = cfg['segmentation'], cfg['snapshot']['tag']
    eps, out = sc['zero_remainder_eps_m'], {}
    safe = pieces[~pieces['unsafe'].astype(bool)]
    for R in sc['resolutions_m']:
        rows = []
        for r in safe.itertuples(index=False):
            xy = np.asarray(r.geom_utm.coords, float)
            L = float(np.hypot(np.diff(xy[:, 0]), np.diff(xy[:, 1])).sum())
            n = n_segments(L, R, eps)
            cuts = [float(i * R) for i in range(n)] + [L]
            short = cfg['sources'][r.source_id]['short']
            for i, c in enumerate(cut_line(xy, cuts)):
                ll = project_xy(inv, c)
                rows.append({'segment_id': segment_id(sc['id_version'], tag, short, r.oid, r.part, r.piece, R, i), 'resolution_m': R, 'snapshot_tag': tag,
                             'source_id': r.source_id, 'oid': int(r.oid), 'part': int(r.part), 'piece': int(r.piece), 'piece_uid': int(r.piece_uid),
                             'seg_index': i, 'n_in_piece': n, 'start_m': cuts[i], 'end_m': cuts[i + 1],
                             'length_m': float(np.hypot(np.diff(c[:, 0]), np.diff(c[:, 1])).sum()), 'length_geod_m': geod_length(ll),
                             'short_final': bool(i == n - 1 and cuts[i + 1] - cuts[i] < R), 'clipped_at_start': bool(i == 0 and r.clipped_at_start),
                             'clipped_at_end': bool(i == n - 1 and r.clipped_at_end), 'piece_overlap': bool(getattr(r, 'overlap_flag', False)),
                             'piece_duplicate': bool(getattr(r, 'duplicate_flag', False)), 'route_raw': r.route_raw, 'route_key': r.route_key,
                             'route_status': r.route_status, 'coords_utm': c, 'coords_ll': ll})
        out[R] = pd.DataFrame(rows, columns=SEG_COLS)
    return out


def locate_events(xy, idx, part_uids, parts, pieces, cfg):
    # offset along the chosen part, the clipped piece holding the nearest point, and one segment ID per resolution (unknown stays None)
    from shapely.geometry import Point
    n, res = len(xy), cfg['segmentation']['resolutions_m']
    eps, tol = cfg['segmentation']['zero_remainder_eps_m'], cfg['matching']['piece_match_tol_m']
    off_part, piece_no, off_piece = np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)
    seg_status = np.full(n, 'not_matched', dtype=object)
    sids = {R: np.full(n, None, dtype=object) for R in res}
    by_part = {k: g for k, g in pieces.groupby('part_uid')}
    tag, ver = cfg['snapshot']['tag'], cfg['segmentation']['id_version']
    for i in idx:
        g = parts['geom_utm'].iloc[int(part_uids[i])]
        s = g.project(Point(xy[i]))
        q = g.interpolate(s)
        off_part[i] = s
        best = None
        grp = by_part.get(int(part_uids[i]))
        for r in ([] if grp is None else grp.itertuples(index=False)):
            dd = r.geom_utm.distance(q)
            if dd <= tol and (best is None or dd < best[0]):
                best = (dd, r)
        if best is None:
            seg_status[i] = 'nearest_point_outside_envelope_clip'
            continue
        r = best[1]
        t = r.geom_utm.project(q)
        piece_no[i], off_piece[i], seg_status[i] = r.piece, t, 'assigned'
        for R in res:
            sids[R][i] = segment_id(ver, tag, cfg['sources'][r.source_id]['short'], r.oid, r.part, r.piece, R, segment_index(t, r.len_utm_m, R, eps))
    out = pd.DataFrame({'offset_on_part_m': off_part, 'chosen_piece': pd.Series(piece_no).astype('Int64'), 'offset_on_piece_m': off_piece,
                        'segment_status': seg_status})
    for R in res:
        out['segment_id_' + str(R) + 'm'] = sids[R]
    return out


def link_events(ev, parts, pieces, cfg, fwd, tols):
    ev = ev.reset_index(drop=True)
    m = cfg['matching']
    margin = float(m['ambiguity_margin_m'])
    pilot = np.flatnonzero((ev['scope'] == 'pilot').to_numpy())
    xy = np.full((len(ev), 2), np.nan)
    if len(pilot):
        xy[pilot] = project_xy(fwd, np.column_stack([ev['lon'].to_numpy(float)[pilot], ev['lat'].to_numpy(float)[pilot]]))
    geoms = parts['geom_utm'].tolist()
    pr = candidate_pairs(xy[pilot], geoms, max(tols) + margin)
    pr['ev'] = pilot[pr['ev'].to_numpy(int)] if len(pr) else pr['ev']
    out, chosen = assign_status(ev, pr, parts, tols, margin)
    d_any, i_any = nearest_within(xy[pilot], geoms, m['diagnostic_nearest_max_m'])
    na, ns = np.full(len(ev), np.nan), np.full(len(ev), None, dtype=object)
    src = parts['source_id'].to_numpy(object)
    na[pilot] = d_any
    ns[pilot] = [src[k] if k >= 0 else None for k in i_any]
    out['nearest_any_road_m'], out['nearest_any_road_source'] = na, ns
    same_src = np.full(len(ev), np.nan)
    evsrc = ev['source_id'].to_numpy(object)
    for sid in cfg['sources']:
        sel = pilot[evsrc[pilot] == sid]
        mask = (parts['source_id'] == sid).to_numpy()
        same_src[sel] = nearest_within(xy[sel], [g for g, k in zip(geoms, mask) if k], m['diagnostic_nearest_max_m'])[0]
    out['nearest_same_agency_any_route_m'] = same_src
    matched_any, first_t = np.zeros(len(ev), bool), np.full(len(ev), np.nan)
    for t in sorted(tols, reverse=True):
        mm = (out['status_' + tol_label(t)] == 'matched').to_numpy()
        first_t[mm] = t
        matched_any |= mm
    out['first_matched_tolerance_m'] = first_t
    out['chosen_source_oid'] = pd.Series(np.where(matched_any, chosen['oid'].to_numpy(float), np.nan)).astype('Int64').to_numpy()
    out['chosen_part'] = pd.Series(np.where(matched_any, chosen['part'].to_numpy(float), np.nan)).astype('Int64').to_numpy()
    out['chosen_distance_m'] = np.where(matched_any, chosen['dist_m'].to_numpy(float), np.nan)
    out['chosen_second_distance_m'] = np.where(matched_any, out['second_same_identity_m'].to_numpy(float), np.nan)
    loc = locate_events(xy, np.flatnonzero(matched_any), chosen['part_uid'].to_numpy(float), parts, pieces, cfg)
    return pd.concat([out, loc], axis=1)

In [ ]:
# FUNCTIONS (4/4): checks, aggregated tables, deterministic review sample, panel sizes (never materialized), figures, Thai report and manifest.
CAVEATS = [
    'descriptive linkage evidence only: no model, probability, threshold, risk score or coloured warning is produced',
    'road geometry is the current service snapshot (task 013a capture); it does not prove 2021-2025 topology; segment IDs are valid only within the pinned snapshot',
    'event coordinates are assumed WGS84 without verification; no positional accuracy and no point-in-province validation is claimed (no province polygons registered)',
    'the envelope is a fixed predeclared scope filter, not an SPU-Khao Yai route, not all six provinces, and never road validation',
    'no record is not proof of no accident; unmatched, ambiguous, identity-unverified, low-precision and outside events are unknown, never zero',
    'linked counts are recorded+linked database counts conditional on the fixed collection and linkage pipeline, not physical accident counts or reporting completeness',
    'EXAT accident route codes have no verified crosswalk to geometry routes: every EXAT event stays identity_unresolved',
    'DRR special codes are unverified and never matched; no alias, padding, substring or fuzzy matching is used',
    'raw KM is diagnostic only (units unverified) and never a match criterion',
    'segment lengths are EPSG:32647 projected metres; projected vs geodesic distortion is reported, not corrected',
    'overlapping/duplicate road pieces are flagged, not merged: inventory length is not unique network mileage and no routable graph is built',
    'V2 province models are not claimed to transfer to points or segments',
]
FIG_COLORS = ['#4c72b0', '#8172b2', '#64b5cd', '#937860', '#8c8c8c', '#c9c9c9', '#2f4b7c', '#a3a3d1', '#5f9ea0', '#b0a18f', '#6a6a6a', '#dddddd', '#1f3a5f', '#7b68ee']


def chk(name, ok, evidence=None):
    return {'check': name, 'status': 'pass' if ok else 'fail', 'evidence': jsonable(evidence)}


def primary_label(cfg):
    return tol_label(cfg['matching']['primary_tolerance_m'])


def geometry_checks(features, parts, pieces, clip_ids, review, cfg, overlap_failed):
    tol, out = cfg['segmentation']['geodesic_crosscheck_tol_m'], []
    dfull = (features['len_full_geod_m'] - features['diag_length_full_m']).abs()
    dclip = (features['len_clipped_geod_m'] - features['diag_length_clipped_m']).abs()
    out.append(chk('full-section geodesic length (pyproj Geod) equals accepted diagnostics within ' + str(tol) + ' m', bool((dfull <= tol).all()),
                   {'features': len(features), 'max_abs_diff_m': float(dfull.max()) if len(dfull) else 0.0}))
    out.append(chk('per-part envelope clip reproduces accepted clipped lengths within ' + str(tol) + ' m', bool((dclip <= tol).all()),
                   {'max_abs_diff_m': float(dclip.max()) if len(dclip) else 0.0}))
    ours = {sid: sorted(int(o) for o in features.loc[(features['source_id'] == sid) & (features['n_pieces'] > 0), 'oid']) for sid in cfg['sources']}
    out.append(chk('features with a non-empty clip equal the accepted clipped GeoJSON ids', ours == {k: list(v) for k, v in clip_ids.items()},
                   {sid: len(v) for sid, v in ours.items()}))
    ns = {sid: int((~features.loc[features['source_id'] == sid, 'simple']).sum()) for sid in cfg['sources']}
    rv = {sid: int(review['counts'][sid]['not_simple']) for sid in cfg['sources']}
    out.append(chk('non-simple feature counts equal the independent review (kept in audit, excluded from segmentation)', ns == rv, {'observed': ns, 'review': rv}))
    bad = int((pieces['unsafe'] & ~pieces['oid'].isin(features.loc[features['unsafe'], 'oid'])).sum())
    out.append(chk('unsafe flag is feature-level and consistent across parts/pieces', bad == 0, {'inconsistent_pieces': bad}))
    out.append(chk('pairwise overlap computation completed for every intersecting piece pair', overlap_failed == 0, {'failed_pairs': overlap_failed}))
    return out


def event_checks(E, cfg, tols, expect):
    out, n = [], len(E)
    out.append(chk('lineage rows equal the accepted readiness manifest', n == expect['lineage_rows'], {'rows': n, 'manifest': expect['lineage_rows']}))
    out.append(chk('event_uid unique (one row per event)', bool(E['event_uid'].is_unique), {'rows': n}))
    for split, g in [('all', E)] + list(E.groupby('split')):
        sc = g['scope'].value_counts().to_dict()
        out.append(chk('scope reconciliation (' + split + '): pilot + outside_envelope + coord_not_usable == retained',
                       sum(sc.values()) == len(g) and set(sc) <= {'pilot', 'outside_envelope', 'coord_not_usable'}, dict(sc, retained=len(g))))
    pilot, outside = (E['scope'] == 'pilot').to_numpy(), ['coord_not_usable', 'outside_pilot_envelope']
    prev = None
    for t in tols:
        lab = tol_label(t)
        st = E['status_' + lab]
        ok = bool(st.isin(STATUSES).all()) and not st[pilot].isin(outside).any() and bool(st[~pilot].isin(outside).all())
        out.append(chk('status reconciliation at ' + lab + ': one status per retained event, pilot statuses sum to pilot', ok,
                       dict(st.value_counts().to_dict(), pilot=int(pilot.sum()), retained=n)))
        if prev is not None:
            mono = bool((E['n_eligible_' + lab] >= E['n_eligible_' + prev]).all())
            mp, mn = E['status_' + prev] == 'matched', st == 'matched'
            out.append(chk('eligible candidate counts monotone ' + prev + ' -> ' + lab, mono))
            out.append(chk('matched set nested ' + prev + ' -> ' + lab, bool((~mp | mn).all()), {'matched_' + prev: int(mp.sum()), 'matched_' + lab: int(mn.sum())}))
        prev = lab
    any_m = E[['status_' + tol_label(t) for t in tols]].eq('matched').any(axis=1).to_numpy()
    has = E['chosen_source_oid'].notna().to_numpy()
    out.append(chk('an accepted road part is recorded iff matched (at most one per event)', bool((any_m == has).all()), {'matched_any': int(any_m.sum())}))
    segc = [c for c in E.columns if c.startswith('segment_id_')]
    num = ['chosen_distance_m', 'chosen_second_distance_m', 'offset_on_part_m', 'offset_on_piece_m']
    zero_free = bool(E.loc[~any_m, num].isna().all().all()) and bool(E.loc[~any_m, segc].isna().all().all())
    out.append(chk('unknown never zero: non-matched rows carry no distance/offset/segment (missing, not 0)', zero_free, {'unknown_rows': int((~any_m).sum())}))
    low = (E['precision_class'] == 'low_precision_unsuitable').to_numpy()
    out.append(chk('low-precision / lat==lon coordinates are never matched', not (low & any_m).any(), {'low_precision_events': int(low.sum())}))
    unres = [sid for sid, s in cfg['sources'].items() if s['normalizer'] == 'exat_unresolved']
    ex = E['source_id'].isin(unres).to_numpy()
    out.append(chk('identity-unresolved agencies (EXAT) are never matched', not (ex & any_m).any(), {'events': int(ex.sum())}))
    return out


def segment_checks(segs, pieces, E, cfg):
    tol, out = cfg['segmentation']['length_conservation_tol_m'], []
    safe = pieces[~pieces['unsafe']]
    for R, S in segs.items():
        tot = S.groupby('piece_uid')['length_m'].sum().reindex(safe['piece_uid']).to_numpy(float)
        diff = np.abs(tot - safe['len_utm_m'].to_numpy(float))
        worst = float(np.max(diff)) if len(diff) else 0.0
        out.append(chk('length conservation per clipped piece < ' + str(tol) + ' m at ' + str(R) + ' m', bool(np.isfinite(diff).all() and worst < tol),
                       {'pieces': len(diff), 'max_abs_diff_m': worst}))
        out.append(chk('segment IDs unique at ' + str(R) + ' m', bool(S['segment_id'].is_unique), {'segments': len(S)}))
        on_unsafe = int(S['piece_uid'].isin(pieces.loc[pieces['unsafe'], 'piece_uid']).sum())
        out.append(chk('no segment built on unsafe geometry at ' + str(R) + ' m', on_unsafe == 0, {'segments_on_unsafe': on_unsafe}))
        within = (S['end_m'] - S['start_m']).to_numpy(float)
        ok = len(S) == 0 or bool(np.abs(S['length_m'].to_numpy(float) - within).max() < 1e-6 and (within <= R + 1e-9).all())
        out.append(chk('segment length equals its interval and never exceeds the target at ' + str(R) + ' m', ok))
        col = 'segment_id_' + str(R) + 'm'
        a = E.loc[E[col].notna(), [col, 'chosen_source_oid']]
        missing = int((~a[col].isin(set(S['segment_id']))).sum())
        oid_ok = bool((a[col].str.split(':').str[2].astype(int).to_numpy() == a['chosen_source_oid'].astype(int).to_numpy()).all()) if len(a) else True
        out.append(chk('every assigned event segment exists in the geometry inventory and belongs to the chosen OID at ' + str(R) + ' m',
                       missing == 0 and oid_ok, {'assigned_events': len(a), 'missing': missing}))
    return out


def count_by(df, cols):
    return df.groupby(cols, dropna=False).size().rename('n').reset_index()


def reconciliation(E):
    rows = []
    for split, g in [('all', E)] + list(E.groupby('split')):
        rows.append({'dimension': 'retained', 'value': 'all', 'split': split, 'n': len(g)})
        rows += [{'dimension': 'scope', 'value': k, 'split': split, 'n': int(v)} for k, v in g['scope'].value_counts().sort_index().items()]
        nu = g[g['scope'] == 'coord_not_usable']['coord_status'].value_counts().sort_index()
        rows += [{'dimension': 'coord_not_usable_by_coord_status', 'value': k, 'split': split, 'n': int(v)} for k, v in nu.items()]
        pa = g[g['scope'] == 'pilot']['raw_agency'].value_counts().sort_index()
        rows += [{'dimension': 'pilot_raw_agency_exact', 'value': k, 'split': split, 'n': int(v)} for k, v in pa.items()]
        dr = g[g['scope'] == 'pilot']['event_date']
        rows.append({'dimension': 'pilot_event_date_range', 'value': (str(dr.min()) + '..' + str(dr.max())) if len(dr) else '', 'split': split, 'n': len(dr)})
    return pd.DataFrame(rows)


def status_long(E, tols):
    frames = []
    for t in tols:
        g = count_by(E, ['split', 'ce_year', 'source_label', 'precision_class', 'status_' + tol_label(t)]).rename(columns={'status_' + tol_label(t): 'status'})
        g.insert(0, 'tolerance_m', t)
        frames.append(g)
    return pd.concat(frames, ignore_index=True)


def status_summary(E, cfg, tols):
    rows, prim = [], float(cfg['matching']['primary_tolerance_m'])
    for split, g in [('all', E)] + list(E.groupby('split')):
        nr, npl = len(g), int((g['scope'] == 'pilot').sum())
        for t in tols:
            vc = g['status_' + tol_label(t)].value_counts()
            for s in STATUSES:
                k = int(vc.get(s, 0))
                in_pilot = s not in ('coord_not_usable', 'outside_pilot_envelope')
                rows.append({'split': split, 'tolerance_m': t, 'primary': t == prim, 'status': s, 'n': k, 'denominator_retained': nr,
                             'share_of_retained': k / nr if nr else None, 'denominator_pilot': npl if in_pilot else None,
                             'share_of_pilot': (k / npl if npl else None) if in_pilot else None})
    return pd.DataFrame(rows)


def distance_distribution(E, cfg, tols):
    radius = max(tols) + float(cfg['matching']['ambiguity_margin_m'])
    sel = E[(E['scope'] == 'pilot') & (E['identity_status'] == 'verified') & (E['precision_class'] == 'adequate_unverified')]
    edges, rows = np.arange(0.0, radius + 5.0, 5.0), []
    for (src, split), g in sel.groupby(['source_label', 'split']):
        d = g['best_same_identity_m'].to_numpy(float)
        h, _ = np.histogram(d[np.isfinite(d)], bins=edges)
        rows += [{'source_label': src, 'split': split, 'bin_lo_m': lo, 'bin_hi_m': hi, 'n': int(c), 'denominator_verified_identity_adequate_precision': len(g)}
                 for lo, hi, c in zip(edges[:-1], edges[1:], h)]
        rows.append({'source_label': src, 'split': split, 'bin_lo_m': radius, 'bin_hi_m': None, 'n': int((~np.isfinite(d)).sum()),
                     'denominator_verified_identity_adequate_precision': len(g)})
    return pd.DataFrame(rows, columns=['source_label', 'split', 'bin_lo_m', 'bin_hi_m', 'n', 'denominator_verified_identity_adequate_precision'])


def km_diagnostic(E, feats, cfg):
    # raw KM never decides a match; this only tests unit hypotheses for matched events against section km_start/km_end
    p, lab = E[E['scope'] == 'pilot'], primary_label(cfg)
    rows = [{'table': 'km_status_counts', 'source_label': s, 'km_status': k, 'hypothesis': None, 'n': int(v), 'n_evaluable': None, 'n_within': None}
            for (s, k), v in p.groupby(['source_label', 'km_status']).size().items()]
    rng = {(f['source_id'], f['oid']): (f['props'].get('km_start'), f['props'].get('km_end')) for f in feats}
    m = p[p['status_' + lab] == 'matched']
    for sid, g in m.groupby('source_id'):
        lohi = [rng.get((sid, int(o)), (None, None)) for o in g['chosen_source_oid']]
        lo = pd.to_numeric(pd.Series([x[0] for x in lohi], dtype=object), errors='coerce').to_numpy(float)
        hi = pd.to_numeric(pd.Series([x[1] for x in lohi], dtype=object), errors='coerce').to_numpy(float)
        v = pd.to_numeric(g['km_value'], errors='coerce').to_numpy(float) * 1000.0
        ok = np.isfinite(v) & np.isfinite(lo) & np.isfinite(hi)
        for unit, scale in cfg['km_diagnostic']['unit_hypotheses'].items():
            a, b = np.minimum(lo, hi) * scale, np.maximum(lo, hi) * scale
            with np.errstate(invalid='ignore'):
                w = ok & (v >= a) & (v <= b)
            rows.append({'table': 'matched_raw_km_within_section_range', 'source_label': sid, 'km_status': 'any', 'hypothesis': 'section km fields in ' + unit,
                         'n': len(g), 'n_evaluable': int(ok.sum()), 'n_within': int(w.sum())})
    return pd.DataFrame(rows)


def name_proposals(E, feats, cfg):
    # EXAT: exact (NFKC, stripped) equality of the raw route name with a geometry name is listed as an UNADOPTED proposal only
    rows = []
    for sid, s in cfg['sources'].items():
        if s['normalizer'] != 'exat_unresolved':
            continue
        names = {}
        for f in feats:
            if f['source_id'] == sid:
                names.setdefault(_nfkc(f['props'].get(s.get('name_field'), '')).strip(), []).append(str(f['oid']) + ':' + str(f['route_raw']))
        g = E[(E['scope'] == 'pilot') & (E['source_id'] == sid)]
        for (code, name), h in g.groupby(['raw_route_code', 'raw_route_name']):
            eq = names.get(_nfkc(name).strip(), []) if _nfkc(name).strip() else []
            rows.append({'source_id': sid, 'raw_route_code': code, 'raw_route_name': name, 'n_pilot': len(h), 'n_train': int((h['split'] == 'train').sum()),
                         'exact_name_equal_geometry': ';'.join(eq), 'decision': 'unadopted_proposal' if eq else 'no_proposal', 'matched': False})
    return pd.DataFrame(rows, columns=['source_id', 'raw_route_code', 'raw_route_name', 'n_pilot', 'n_train', 'exact_name_equal_geometry', 'decision', 'matched'])


def special_codes(E, features):
    g = E[(E['scope'] == 'pilot') & (E['identity_status'] == 'route_special_unverified')]
    geo = set(zip(features.loc[features['route_status'] == 'route_special_unverified', 'source_id'], features.loc[features['route_status'] == 'route_special_unverified', 'route_key']))
    t = count_by(g, ['source_label', 'route_normalized'])
    t['same_string_in_geometry'] = [(s, r) in geo for s, r in zip(t['source_label'], t['route_normalized'])]
    t['decision'] = 'unverified_not_matched'
    return t


def geometry_audit(features, pieces):
    rows = []
    for sid, f in features.groupby('source_id'):
        c = pieces[pieces['source_id'] == sid]
        u, s = f[f['unsafe']], c[~c['unsafe']]
        reasons = pd.Series([r for x in u['unsafe_reasons'] for r in x.split(';') if r]).value_counts().to_dict()
        rows.append({'source_id': sid, 'features': len(f), 'multipart_features': int((f['n_parts'] > 1).sum()), 'features_with_clip': int((f['n_pieces'] > 0).sum()),
                     'unsafe_features': len(u), 'unsafe_reasons': ';'.join(k + '=' + str(v) for k, v in sorted(reasons.items())),
                     'unsafe_full_geod_km': u['len_full_geod_m'].sum() / 1000, 'unsafe_clipped_geod_km': u['len_clipped_geod_m'].sum() / 1000,
                     'pieces_total': len(c), 'pieces_usable': len(s), 'usable_clipped_utm_km': s['len_utm_m'].sum() / 1000,
                     'usable_clipped_geod_km': s['len_geod_m'].sum() / 1000, 'usable_pieces_cut_at_envelope': int((s['clipped_at_start'] | s['clipped_at_end']).sum()),
                     'route_status_counts': ';'.join(k + '=' + str(v) for k, v in f['route_status'].value_counts().sort_index().items())})
    return pd.DataFrame(rows)


def projection_distortion(pieces):
    s = pieces[(~pieces['unsafe']) & (pieces['len_geod_m'] > 0)]
    rows = []
    for sid, g in [('all', s)] + list(s.groupby('source_id')):
        r = (g['len_utm_m'] / g['len_geod_m']).to_numpy(float)
        rows.append({'source_id': sid, 'pieces': len(g), 'ratio_min': r.min() if len(r) else None, 'ratio_median': float(np.median(r)) if len(r) else None,
                     'ratio_max': r.max() if len(r) else None, 'max_abs_ppm': float(np.abs(r - 1).max() * 1e6) if len(r) else None,
                     'total_utm_km': g['len_utm_m'].sum() / 1000, 'total_geod_km': g['len_geod_m'].sum() / 1000})
    return pd.DataFrame(rows)


def inventory_summary(segs):
    rows = []
    for R, S in segs.items():
        for sid, g in [('all', S)] + list(S.groupby('source_id')):
            rows.append({'resolution_m': R, 'source_id': sid, 'segments': len(g), 'full_length_segments': int((~g['short_final']).sum()),
                         'short_final_segments': int(g['short_final'].sum()), 'pieces': int(g['piece_uid'].nunique()), 'route_keys': int(g['route_key'].nunique()),
                         'total_utm_km': g['length_m'].sum() / 1000, 'total_geod_km': g['length_geod_m'].sum() / 1000,
                         'median_length_m': float(g['length_m'].median()) if len(g) else None, 'min_length_m': float(g['length_m'].min()) if len(g) else None,
                         'segments_cut_at_envelope': int((g['clipped_at_start'] | g['clipped_at_end']).sum()), 'segments_on_overlapping_piece': int(g['piece_overlap'].sum()),
                         'segments_on_duplicate_piece': int(g['piece_duplicate'].sum())})
    return pd.DataFrame(rows)


def split_days(cfg, split):
    ys = cfg['years'][split]
    start, end = pd.Timestamp(min(ys), 1, 1), pd.Timestamp(max(ys), 12, 31)
    return start, end, int((end - start).days) + 1, len(pd.period_range(start, end, freq='W-MON'))


def panel_estimates(segs, cfg):
    # sizes only: no panel is materialized and no zero is filled
    rows, blocks = [], cfg['panel_estimate']['blocks_per_day_6h']
    for split in ('train', 'seen_diagnostic'):
        start, end, days, weeks = split_days(cfg, split)
        for R, S in segs.items():
            n = len(S)
            rows.append({'resolution_m': R, 'split': split, 'date_start': str(start.date()), 'date_end': str(end.date()), 'candidate_segments': n,
                         'days': days, 'weeks_w_mon': weeks, 'rows_daily': n * days, 'rows_6h': n * days * blocks, 'rows_weekly': n * weeks, 'materialized': False})
    return pd.DataFrame(rows)


def train_support(E, segs, cfg):
    # TRAIN ONLY (2021-2023): support of recorded+linked counts; seen 2024/25 outcomes are not used to compare resolutions
    lab = primary_label(cfg)
    tr = E[(E['split'] == 'train') & (E['status_' + lab] == 'matched') & (E['segment_status'] == 'assigned')]
    _, _, days, weeks = split_days(cfg, 'train')
    rows = []
    for R, S in segs.items():
        col, n = 'segment_id_' + str(R) + 'm', len(S)
        per = tr[col].value_counts()
        sd = tr.groupby([col, 'event_date']).size()
        rows.append({'resolution_m': R, 'split': 'train', 'candidate_segments': n, 'linked_train_events': len(tr), 'segments_with_any_linked_event': int(per.size),
                     'share_segments_with_any': per.size / n if n else None, 'mean_linked_events_per_segment': len(tr) / n if n else None,
                     'median_events_where_nonzero': float(per.median()) if per.size else None, 'max_events_one_segment': int(per.max()) if per.size else 0,
                     'segment_days_with_any': int(sd.size), 'share_segment_days_with_any': sd.size / (n * days) if n else None,
                     'max_events_one_segment_day': int(sd.max()) if sd.size else 0,
                     'note': 'recorded+linked counts conditional on the pipeline; cells without a linked record are unknown, not verified absence'})
    return pd.DataFrame(rows)


def review_strata(E, cfg):
    lab, hi = primary_label(cfg), cfg['matching']['high_distance_review_m']
    p = E[E['scope'] == 'pilot']
    st = p['status_' + lab]
    with np.errstate(invalid='ignore'):
        far = (p['chosen_distance_m'].to_numpy(float) > hi)
    lab_ = np.select([st.eq('matched').to_numpy() & far, st.eq('matched').to_numpy(), st.isin(['ambiguous_multiple', 'ambiguous_unsafe_geometry']).to_numpy(),
                      st.eq('too_far').to_numpy(), st.eq('low_precision_unsuitable').to_numpy()],
                     ['matched_high_distance', 'matched', 'ambiguous', 'too_far', 'low_precision_unsuitable'], default='identity_unverified')
    return pd.Series(lab_, index=p.index).astype(object) + '|' + p['source_label'].astype(str)


def review_sample(E, cfg):
    # fixed seed, sorted event_uids per stratum, strata in sorted order: deterministic and not hand-picked
    rng, k = np.random.default_rng(cfg['review_sample']['seed']), cfg['review_sample']['per_stratum']
    strata = review_strata(E, cfg)
    groups = [(s, strata.index[(strata == s).to_numpy()]) for s in sorted(strata.unique())]
    rep = E.loc[strata.index]
    groups.append(('repeated_coordinate|any', rep.index[rep['repeated_coord'].to_numpy(bool)]))
    chosen, rows = {}, []
    for s, ix in groups:
        uids = np.array(sorted(E.loc[ix, 'event_uid'].tolist()), dtype=object)
        picked = sorted(uids[rng.choice(len(uids), size=min(k, len(uids)), replace=False)].tolist()) if len(uids) else []
        rows.append({'stratum': s, 'population': len(uids), 'sampled': len(picked)})
        for u in picked:
            chosen.setdefault(u, []).append(s)
    sample = E[E['event_uid'].isin(list(chosen))].copy()
    sample['strata'] = sample['event_uid'].map(lambda u: ';'.join(chosen[u]))
    sample['review_status'] = 'pending_manual_review'
    return pd.DataFrame(rows), sample.sort_values('event_uid').reset_index(drop=True)


def build_tables(E, features, pieces, segs, overlaps, input_rows, feats, cfg, tols):
    T = {'input_verification': input_rows, 'event_reconciliation': reconciliation(E)}
    T['coordinate_flags'] = count_by(E, ['split', 'scope', 'coord_status', 'precision_class', 'low_precision', 'lat_equals_lon', 'repeated_coord'])
    T['identity_rules'] = pd.DataFrame([{'source_id': sid, 'agency_exact': s['agency_exact'], 'normalizer': s['normalizer'], 'rule': s['rule'],
                                         'alias_entries': len((cfg.get('alias_map') or {}).get(sid, {}))} for sid, s in cfg['sources'].items()])
    T['identity_by_split'] = count_by(E, ['scope', 'split', 'source_label', 'identity_status'])
    T['geometry_route_identity'] = features.groupby(['source_id', 'route_status']).agg(features=('oid', 'size'), parts=('n_parts', 'sum')).reset_index()
    nis = E[(E['scope'] == 'pilot') & (E['identity_status'] == 'route_not_in_snapshot')]
    T['route_not_in_snapshot_top'] = count_by(nis, ['source_label', 'route_normalized']).sort_values(['source_label', 'n'], ascending=[True, False]).groupby('source_label').head(15)
    T['special_codes'] = special_codes(E, features)
    T['exat_name_proposals'] = name_proposals(E, feats, cfg)
    T['linkage_status_by_tolerance'] = status_long(E, tols)
    T['linkage_status_summary'] = status_summary(E, cfg, tols)
    T['distance_distribution_primary'] = distance_distribution(E, cfg, tols)
    T['flags_by_status_primary'] = count_by(E[E['scope'] == 'pilot'], ['status_' + primary_label(cfg), 'repeated_coord', 'cross_identity_within_margin', 'segment_status'])
    T['km_diagnostic'] = km_diagnostic(E, feats, cfg)
    T['source_geometry_audit'] = geometry_audit(features, pieces)
    T['geometry_exclusions'] = features.loc[features['unsafe'], ['source_id', 'oid', 'n_parts', 'unsafe_reasons', 'len_full_geod_m', 'len_clipped_geod_m']].reset_index(drop=True)
    T['projection_distortion'] = projection_distortion(pieces)
    T['overlap_summary'] = (overlaps.groupby(['source_a', 'source_b', 'exact_duplicate']).agg(pairs=('overlap_m', 'size'), overlap_km=('overlap_m', lambda v: v.sum() / 1000)).reset_index()
                            if len(overlaps) else pd.DataFrame(columns=['source_a', 'source_b', 'exact_duplicate', 'pairs', 'overlap_km']))
    T['segment_inventory_summary'] = inventory_summary(segs)
    T['panel_size_estimates'] = panel_estimates(segs, cfg)
    T['train_support_diagnostics'] = train_support(E, segs, cfg)
    return T


def make_figures(T, fig_dir, cfg, tols):
    # descriptive linkage evidence; neutral categorical colours (no traffic-light / risk palette)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    names = {f.name for f in font_manager.fontManager.ttflist}
    font = next((n for n in cfg['figure_fonts'] if n in names), None)
    if font:
        plt.rcParams['font.family'] = font
    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    prim, lab, files = float(cfg['matching']['primary_tolerance_m']), primary_label(cfg), []
    tag = 'Descriptive linkage evidence - NOT a forecast or risk level'
    L = T['linkage_status_by_tolerance']
    L = L[(L['tolerance_m'] == prim) & ~L['status'].isin(['coord_not_usable', 'outside_pilot_envelope'])]
    piv = L.groupby(['source_label', 'status'])['n'].sum().unstack(fill_value=0)
    piv = piv[[s for s in STATUSES if s in piv.columns]]
    tot = piv.sum(axis=1)
    fig, ax = plt.subplots(figsize=(11, 4.8))
    left = np.zeros(len(piv))
    for i, s in enumerate(piv.columns):
        share = (piv[s] / tot.where(tot > 0)).fillna(0).to_numpy()
        ax.barh(range(len(piv)), share, left=left, color=FIG_COLORS[i % len(FIG_COLORS)], label=s, edgecolor='white', linewidth=0.5)
        left += share
    ax.set_yticks(range(len(piv)))
    ax.set_yticklabels([str(i) + ' (n=' + format(int(tot[i]), ',') + ')' for i in piv.index])
    ax.set_xlim(0, 1)
    ax.set_xlabel('share of pilot-envelope events (denominator n per agency label)')
    ax.set_title('Linkage status at ' + lab + ' (2021-2025 retained events inside envelope)' + NL + tag, fontsize=10)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=4, fontsize=8, frameon=False)
    fig.tight_layout()
    files.append(fig_dir / 'linkage_status_primary.png')
    fig.savefig(files[-1], dpi=130, bbox_inches='tight')
    plt.close(fig)
    D = T['distance_distribution_primary']
    D = D[D['bin_hi_m'].notna()]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for i, (src, g) in enumerate(D.groupby('source_label')):
        h = g.groupby('bin_lo_m')['n'].sum()
        ax.step(h.index.to_numpy(float), h.to_numpy(), where='post', color=FIG_COLORS[i], label=src + ' (n=' + format(int(h.sum()), ',') + ' within radius)')
    for t in tols:
        ax.axvline(t, color='#555555', linestyle='--' if t == prim else ':', linewidth=1)
        ax.text(t, ax.get_ylim()[1] * 0.95, ' ' + tol_label(t), fontsize=8, color='#333333')
    ax.set_xlabel('projected distance to nearest same-identity road part (m, EPSG:32647; event datum assumed WGS84)')
    ax.set_ylabel('events per 5 m bin')
    ax.set_title('Distance distribution, verified route identity, adequate precision' + NL + tag, fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()
    files.append(fig_dir / 'distance_distribution_primary.png')
    fig.savefig(files[-1], dpi=130, bbox_inches='tight')
    plt.close(fig)
    S = T['segment_inventory_summary']
    S = S[S['source_id'] != 'all']
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    res, srcs = sorted(S['resolution_m'].unique()), sorted(S['source_id'].unique())
    w = 0.8 / max(1, len(srcs))
    for i, sid in enumerate(srcs):
        g = S[S['source_id'] == sid].set_index('resolution_m').reindex(res)
        x = np.arange(len(res)) + i * w
        axes[0].bar(x, g['segments'].fillna(0), width=w, color=FIG_COLORS[i], label=sid)
        axes[1].bar(x, (g['short_final_segments'] / g['segments']).fillna(0), width=w, color=FIG_COLORS[i], label=sid)
    for ax, yl in ((axes[0], 'candidate segments (usable geometry)'), (axes[1], 'share short final segments')):
        ax.set_xticks(np.arange(len(res)) + w * (len(srcs) - 1) / 2)
        ax.set_xticklabels([str(r) + ' m' for r in res])
        ax.set_ylabel(yl)
    axes[0].legend(fontsize=8, frameon=False)
    fig.suptitle('Candidate segment inventory from road geometry (includes roads without records)' + NL + tag, fontsize=10)
    fig.tight_layout()
    files.append(fig_dir / 'segment_inventory.png')
    fig.savefig(files[-1], dpi=130, bbox_inches='tight')
    plt.close(fig)
    return files


def _cell(v):
    if v is None or v is pd.NA or (isinstance(v, (float, np.floating)) and not math.isfinite(float(v))):
        return ''
    if isinstance(v, (float, np.floating)):
        v = float(v)
        return str(int(v)) if v.is_integer() and abs(v) < 1e15 else (format(v, '.1f') if abs(v) >= 1000 else format(v, '.4g'))
    return str(v).replace('|', '/')


def md_table(df, max_rows=40):
    if df is None or len(df) == 0:
        return '_(ไม่มีแถว)_'
    d = df.head(max_rows)
    lines = ['| ' + ' | '.join(str(c) for c in d.columns) + ' |', '|' + '---|' * len(d.columns)]
    lines += ['| ' + ' | '.join(_cell(v) for v in r) + ' |' for r in d.itertuples(index=False)]
    return NL.join(lines)


def run_commands():
    return ['.venv/bin/python backend/scripts/run_spatial_linkage_audit.py --output-dir backend/rerun-linkage-014-001',
            '.venv/bin/python -m pytest backend/tests/test_spatial_linkage_audit_notebook.py -q']


def overview_table(E, cfg):
    lab, rows = primary_label(cfg), []
    for split, g in [('all', E)] + [(s, E[E['split'] == s]) for s in ('train', 'seen_diagnostic')]:
        p = g[g['scope'] == 'pilot']
        rows.append({'split': split, 'event_date': (str(g['event_date'].min()) + '..' + str(g['event_date'].max())) if len(g) else '', 'retained': len(g),
                     'pilot_in_envelope': len(p), 'outside_envelope': int((g['scope'] == 'outside_envelope').sum()),
                     'coord_not_usable': int((g['scope'] == 'coord_not_usable').sum()), 'matched_' + lab: int((g['status_' + lab] == 'matched').sum()),
                     'matched_with_segment_' + lab: int(((g['status_' + lab] == 'matched') & (g['segment_status'] == 'assigned')).sum())})
    return pd.DataFrame(rows)


def build_protocol_md(T, cfg):
    P = T['panel_size_estimates']
    L = ['# ข้อเสนอโปรโตคอลขั้นถัดไป (proposal เท่านั้น ยังไม่ดำเนินการ)', '',
         '> หลักฐานเชิงพรรณนาของการเชื่อมโยงข้อมูล ไม่ใช่การพยากรณ์หรือระดับความเสี่ยง ไม่มีการ train/threshold/สีเตือนในขั้นนี้', '',
         '## เป้าหมายการนับที่เสนอ (conditional research count target)',
         '- นับ **จำนวนบันทึกที่ถูกบันทึกและเชื่อมโยงได้ (recorded + linked database count)** ต่อ segment ต่อช่วงเวลา โดยมีเงื่อนไขว่า pipeline การเก็บข้อมูลและ linkage คงที่ตาม config ' + cfg['version'] + ' และ snapshot ' + cfg['snapshot']['tag'],
         '- สิ่งนี้ **ไม่ใช่** จำนวนอุบัติเหตุทางกายภาพ และไม่ใช่ความครบถ้วนของการรายงาน ต้องเปิดเผย ascertainment (ใครรายงาน/บันทึก) และ linkage selection (เหตุที่ unmatched/ambiguous/identity ไม่ยืนยัน/low precision/outside ถูกตัดออก)',
         '- เซลล์ที่ไม่มีบันทึกเชื่อมโยง = ไม่ทราบ/ไม่มีบันทึกที่เชื่อมได้ ไม่ใช่หลักฐานว่าไม่มีอุบัติเหตุ ห้ามเติมศูนย์โดยไม่ระบุสมมติฐานนี้ และต้องรายงานสัดส่วนเหตุที่ไม่ทราบตำแหน่งควบคู่ทุกครั้ง',
         '- สมมติฐาน snapshot: geometry ปัจจุบันใช้แทนโครงข่าย 2021-2025 (ยังไม่พิสูจน์) segment ID ใช้ได้เฉพาะ snapshot นี้',
         '- ไม่อ้างว่าโมเดลระดับจังหวัด V2 ถ่ายทอดสู่ระดับจุด/segment และไม่อ้างความน่าจะเป็นของบุคคล', '',
         '## การเลือก resolution', '- เลือกจาก support diagnostics ของ **train 2021-2023 เท่านั้น** (train_support_diagnostics.csv) และเกณฑ์ที่ลงทะเบียนก่อนดูผล ห้ามใช้ผลลัพธ์ปี 2024/25 (seen) เลือก resolution', '',
         '## ขนาด panel โดยประมาณ (ไม่ได้สร้างจริง)', md_table(P), '',
         '## เงื่อนไขก่อนเริ่ม', '- ทบทวนตัวอย่าง review_sample (data dir, ignored) ด้วยคนแบบ blind ตาม strata ที่กำหนด', '- ยืนยัน crosswalk EXAT และรหัสพิเศษ DRR จากเอกสารทางการก่อนจะนับรวม',
         '- ลงทะเบียนสูตรเป้าหมาย/ช่วงเวลา/ตัวแปร และวิธีจัดการ unknown ก่อนดูผล']
    return NL.join(L) + NL


def build_report_th(E, T, checks, cfg, tols, status, out_root):
    lab = primary_label(cfg)
    S = T['linkage_status_summary']
    npass = sum(c['status'] == 'pass' for c in checks)
    L = ['# รายงานตรวจการเชื่อมโยงอุบัติเหตุกับถนนและคลัง segment (V3 notebook 18)', '',
         '> **หลักฐานเชิงพรรณนาของการเชื่อมโยงข้อมูลเท่านั้น ไม่ใช่การพยากรณ์ ไม่ใช่ระดับความเสี่ยง** ไม่มีการ train โมเดล ไม่มี probability/threshold/สีเตือน', '',
         'สถานะการตรวจรับอัตโนมัติ: **' + status + '** (' + str(npass) + '/' + str(len(checks)) + ' checks pass) · config ' + cfg['version'] + ' · snapshot ' + cfg['snapshot']['tag'], '',
         '## 1. ขอบเขตและวันที่', 'envelope ที่ลงทะเบียนไว้ EPSG4326 ' + str(cfg['envelope_epsg4326']) + ' (ไม่ใช่เส้นทาง SPU-เขาใหญ่ ไม่ครอบคลุม 6 จังหวัด ใช้เป็นตัวกรองขอบเขต ไม่ใช่การยืนยันว่าอยู่บนถนน)', '',
         md_table(overview_table(E, cfg)), '',
         'train = 2021-2023 · seen_diagnostic = 2024-2025 (ข้อมูลที่เคยเห็นแล้ว ใช้รายงาน linkage เท่านั้น ไม่ใช้เลือก resolution)', '',
         '## 2. Input ที่ตรวจ hash แล้ว', md_table(T['input_verification'][['input', 'sha256', 'rule', 'status']]), '',
         '## 3. พิกัด (ไม่ซ่อม)', 'พิกัดที่ swap/หาย/ไม่เป็นตัวเลข/นอกกรอบประเทศ ถูก flag ตามสถานะจาก notebook 16 และไม่ถูกแก้ ค่าความละเอียดต่ำ (ทศนิยม ≤ 3) หรือ lat==lon = ไม่เหมาะกับการกำหนดถนนแบบมั่นใจ', '',
         md_table(T['event_reconciliation'][T['event_reconciliation']['split'] == 'all']), '',
         '## 4. กฎ identity ของสายทาง (exact, ไม่มี alias)', md_table(T['identity_rules']), '',
         md_table(count_by(E[E['scope'] == 'pilot'], ['source_label', 'identity_status'])), '',
         'EXAT: ข้อเสนอชื่อที่ตรงกันแบบ exact (ยังไม่นำมาใช้):', md_table(T['exat_name_proposals'], 15), '',
         '## 5. ผล linkage (primary ' + lab + ', sensitivity ' + ', '.join(tol_label(t) for t in tols if t != float(cfg['matching']['primary_tolerance_m'])) + ')',
         'กฎ: agency ตรง + route ที่ยืนยัน + ระยะ ≤ tolerance; มีคู่แข่ง same-identity ภายใน ' + str(cfg['matching']['ambiguity_margin_m']) + ' ม. ของระยะที่ใกล้สุด = ambiguous_multiple; geometry ไม่ simple/valid = ambiguous_unsafe_geometry; ระยะ diagnostic ไม่ถูกยอมรับโดยปริยาย', '',
         md_table(S[(S['split'] == 'all') & (S['n'] > 0)][['tolerance_m', 'status', 'n', 'denominator_retained', 'share_of_retained', 'denominator_pilot', 'share_of_pilot']], 60), '',
         '## 6. คลัง segment ผู้สมัครจาก geometry', 'ID: ' + cfg['segmentation']['id_version'] + '-<snapshot>:<agency>:<OID>:p<part>:c<clipped piece>:r<target m>:<index> · ' + cfg['segmentation']['endpoint_convention'], '',
         md_table(T['segment_inventory_summary']), '', md_table(T['source_geometry_audit']), '', 'ความเพี้ยนของความยาว UTM เทียบ geodesic:', md_table(T['projection_distortion']), '',
         '## 7. สิ่งที่ทำได้ / ยังทำไม่ได้',
         '- ทำได้: นับบันทึกที่เชื่อมโยงได้อย่างมั่นใจต่อ segment (ภายใต้ pipeline และ snapshot นี้) สำหรับ DOH/DRR ที่ route ยืนยันได้ ภายใน envelope นำร่อง',
         '- ยังทำไม่ได้: EXAT (ไม่มี crosswalk), รหัสพิเศษ DRR, เหตุที่ ambiguous/too_far/low precision/นอก envelope; ไม่มีการพิสูจน์ความครบถ้วนการรายงาน ไม่มี topology ย้อนหลัง ไม่มี datum ที่ยืนยัน',
         '- ไม่มีบันทึก ≠ ไม่มีอุบัติเหตุ; ไม่สร้าง zero-filled panel ไม่มี probability ระดับบุคคล ไม่อ้างว่าโมเดล V2 ระดับจังหวัดใช้กับจุดได้', '',
         '## 8. ขั้นถัดไป', 'ดู next_stage_protocol_proposal.md (ข้อเสนอเท่านั้น)', '',
         '## 9. คำสั่งที่ใช้รัน', NL.join('- `' + c + '`' for c in run_commands()), '', 'output root ของรันนี้: `' + str(out_root) + '`', '',
         '## 10. Checks', md_table(pd.DataFrame([{'check': c['check'], 'status': c['status']} for c in checks]), 100), '',
         '## 11. ข้อจำกัด', NL.join('- ' + c for c in CAVEATS)]
    return NL.join(L) + NL


def segments_geojson(S, cfg):
    props = ['segment_id', 'resolution_m', 'snapshot_tag', 'source_id', 'oid', 'part', 'piece', 'seg_index', 'n_in_piece', 'start_m', 'end_m', 'length_m',
             'length_geod_m', 'short_final', 'clipped_at_start', 'clipped_at_end', 'piece_overlap', 'piece_duplicate', 'route_key', 'route_status']
    feats = [{'type': 'Feature', 'id': r['segment_id'], 'geometry': {'type': 'LineString', 'coordinates': [[float(x), float(y)] for x, y in r['coords_ll']]},
              'properties': {k: jsonable(r[k]) for k in props}} for r in S.to_dict('records')]
    return {'type': 'FeatureCollection', 'note': 'EPSG:4326 lon/lat; start/end/length in EPSG:32647 projected metres; valid only within snapshot ' + cfg['snapshot']['tag'],
            'features': feats}


def write_outputs(paths, E, T, segs, pieces, overlaps, sample, cfg):
    # row-level files and GeoJSON go to the (gitignored) data dir only; the report dir receives aggregates
    d = Path(paths['data_dir'])
    atomic_write_bytes(d / 'event_linkage_audit.csv.gz', csv_bytes(E, gz=True))
    allseg = pd.concat([S.drop(columns=['coords_utm', 'coords_ll']) for S in segs.values()], ignore_index=True)
    atomic_write_bytes(d / 'segment_inventory.csv.gz', csv_bytes(allseg, gz=True))
    for R, S in segs.items():
        atomic_write_bytes(d / ('segments_' + str(R) + 'm.geojson'), json.dumps(segments_geojson(S, cfg), ensure_ascii=False, allow_nan=False).encode('utf-8'))
    atomic_write_bytes(d / 'road_pieces_audit.csv', csv_bytes(pieces.drop(columns=['geom_utm', 'coords_ll'])))
    atomic_write_bytes(d / 'piece_overlaps.csv', csv_bytes(overlaps))
    atomic_write_bytes(d / 'review_sample.csv', csv_bytes(sample))
    rep = Path(paths['report_dir'])
    for name, df in T.items():
        check_report_file(name + '.csv', cfg)
        atomic_write_bytes(rep / (name + '.csv'), csv_bytes(df))
    return True


def hash_dir(d, exclude=()):
    d = Path(d)
    return {q.relative_to(d).as_posix(): sha256_file(q) for q in sorted(d.rglob('*')) if q.is_file() and q.relative_to(d).as_posix() not in exclude}

In [ ]:
# SETUP: config, output guard (before any write), protected snapshot, hash-verified inputs, lineage and road features
CFG = json.loads(CONFIG_PATH.read_text('utf-8'))
CFG_SHA = sha256_file(CONFIG_PATH)
NB_PATH = BACKEND / CFG['notebook']
NB_SHA = sha256_file(NB_PATH) if NB_PATH.is_file() else None
TOLS = validate_config(CFG)
require_modules(['pandas', 'numpy', 'shapely', 'pyproj', 'matplotlib'], CFG['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, CFG)
check_outputs(PATHS, BACKEND, CFG)
STARTED = utcnow()
PROTECTED_BEFORE = hash_paths(BACKEND, CFG['protected_paths'])
INPUT_ROWS, DOCS, EXPECT = verify_inputs(BACKEND, CFG)
LINEAGE = load_lineage(BACKEND / CFG['inputs']['lineage']['path'], EXPECT['lineage_rows'], CFG)
FEATS, DIAG = load_features(BACKEND / CFG['inputs']['road_derived_dir'], CFG, EXPECT['road_counts'])
CLIPPED_IDS = clipped_ids(BACKEND / CFG['inputs']['road_derived_dir'], CFG)
print('inputs verified:', len(INPUT_ROWS), '| lineage rows:', len(LINEAGE), '| road features:', {s: sum(f['source_id'] == s for f in FEATS) for s in CFG['sources']})

In [ ]:
# ROADS: metric parts and envelope-clipped pieces, safety flags, overlaps, route index and geometry checks
FWD, INV = make_transformers(CFG)
FEATURES, PARTS, PIECES = build_roads(FEATS, DIAG, CFG, FWD)
OVERLAPS, OV_FLAGS, OV_FAILED = piece_overlaps(PIECES, CFG['segmentation']['overlap_min_length_m'])
PIECES = pd.concat([PIECES, OV_FLAGS], axis=1)
ROUTE_INDEX = route_index(PARTS)
GEOM_CHECKS = geometry_checks(FEATURES, PARTS, PIECES, CLIPPED_IDS, DOCS['road_review'], CFG, OV_FAILED)
print(FEATURES.groupby('source_id').agg(features=('oid', 'size'), unsafe=('unsafe', 'sum'), pieces=('n_pieces', 'sum')))
print(pd.DataFrame(GEOM_CHECKS)[['check', 'status']].to_string())

In [ ]:
# EVENTS: coordinate flags, exact identity, candidate search and conservative statuses at every preregistered tolerance
COORDS = coordinate_audit(LINEAGE, CFG)
IDENT = event_identity(LINEAGE, CFG, ROUTE_INDEX)
EV = pd.concat([LINEAGE, COORDS, IDENT], axis=1)
LINK = link_events(EV, PARTS, PIECES, CFG, FWD, TOLS)
E = pd.concat([EV, LINK], axis=1)
E['source_label'] = E['source_id'].fillna('unmapped_agency')
print(E['scope'].value_counts().to_string())
print(E[E['scope'] == 'pilot'].groupby(['source_label', 'status_' + primary_label(CFG)]).size().to_string())

In [ ]:
# SEGMENTS: 0.5/1/2 km candidate inventories from usable geometry, then segment and event-assignment checks
SEGS = build_segments(PIECES, CFG, INV)
SEG_CHECKS = segment_checks(SEGS, PIECES, E, CFG)
print({R: len(S) for R, S in SEGS.items()})
print(pd.DataFrame(SEG_CHECKS)[['check', 'status']].to_string())

In [ ]:
# TABLES: aggregated evidence, deterministic review sample, all checks
TABLES = build_tables(E, FEATURES, PIECES, SEGS, OVERLAPS, INPUT_ROWS, FEATS, CFG, TOLS)
STRATA, SAMPLE = review_sample(E, CFG)
TABLES['review_sample_strata'] = STRATA
CHECKS = event_checks(E, CFG, TOLS, EXPECT) + GEOM_CHECKS + SEG_CHECKS
TABLES['checks'] = pd.DataFrame([{'check': c['check'], 'status': c['status'], 'evidence': json.dumps(c['evidence'], ensure_ascii=False)} for c in CHECKS])
print(TABLES['linkage_status_summary'].query('split == ' + repr('all') + ' and n > 0').to_string())
print('checks failing:', [c['check'] for c in CHECKS if c['status'] != 'pass'])

In [ ]:
# OUTPUTS: ignored row-level data, aggregated reports + figures, protected re-hash, then the acceptance manifest (complete only if every check passes)
write_outputs(PATHS, E, TABLES, SEGS, PIECES, OVERLAPS, SAMPLE, CFG)
FIGS = make_figures(TABLES, PATHS['figure_dir'], CFG, TOLS)
PROTECTED_AFTER = hash_paths(BACKEND, CFG['protected_paths'])
CHANGED = sorted(k for k in set(PROTECTED_BEFORE) | set(PROTECTED_AFTER) if PROTECTED_BEFORE.get(k) != PROTECTED_AFTER.get(k))
CHECKS.append(chk('protected inputs/outputs unchanged (sha256 before == after)', not CHANGED, {'files_hashed': len(PROTECTED_AFTER), 'changed': CHANGED[:20]}))
STATUS = 'complete' if all(c['status'] == 'pass' for c in CHECKS) else 'failed'
REPORT_MD = build_report_th(E, TABLES, CHECKS, CFG, TOLS, STATUS, OUTPUT_ROOT)
for name, text in (('linkage_audit_report_th.md', REPORT_MD), ('next_stage_protocol_proposal.md', build_protocol_md(TABLES, CFG))):
    check_report_file(name, CFG)
    atomic_write_bytes(PATHS['report_dir'] / name, text.encode('utf-8'))
lab = primary_label(CFG)
MANIFEST = {
    'status': STATUS, 'version': CFG['version'], 'purpose': 'descriptive accident-to-road linkage audit and candidate segment inventories; NOT a forecast, model or risk level',
    'started_utc': STARTED, 'finished_utc': utcnow(), 'config_sha256': CFG_SHA, 'notebook': CFG['notebook'], 'notebook_sha256_at_start': NB_SHA,
    'output_root': str(OUTPUT_ROOT), 'output_root_is_backend': OUTPUT_ROOT == BACKEND, 'snapshot_tag': CFG['snapshot']['tag'],
    'inputs': INPUT_ROWS.to_dict('records'),
    'environment': {'python': platform.python_version(), 'platform': platform.platform(),
                    'dependencies': {n: module_version(n) for n in ('pandas', 'numpy', 'shapely', 'pyproj', 'matplotlib')}},
    'rules': {'crs': CFG['crs'], 'matching': CFG['matching'], 'segmentation': CFG['segmentation'], 'alias_map_entries': sum(len(v) for v in (CFG.get('alias_map') or {}).values()),
              'sources': {sid: {k: s[k] for k in ('agency_exact', 'normalizer', 'rule')} for sid, s in CFG['sources'].items()}},
    'reconciliation': {'retained': len(E), 'scope': E['scope'].value_counts().to_dict(),
                       'pilot_by_split': E[E['scope'] == 'pilot']['split'].value_counts().to_dict(),
                       'pilot_raw_agency_exact': E[E['scope'] == 'pilot']['raw_agency'].value_counts().to_dict(),
                       'status_by_tolerance': {tol_label(t): E['status_' + tol_label(t)].value_counts().to_dict() for t in TOLS},
                       'segments': {str(R): len(S) for R, S in SEGS.items()}, 'road_features': FEATURES['source_id'].value_counts().to_dict(),
                       'unsafe_features': FEATURES[FEATURES['unsafe']]['source_id'].value_counts().to_dict(),
                       'matched_with_segment_' + lab: int(((E['status_' + lab] == 'matched') & (E['segment_status'] == 'assigned')).sum())},
    'checks': CHECKS,
    'report_files': hash_dir(PATHS['report_dir'], exclude=('linkage_audit_manifest.json',)),
    'data_files_gitignored': hash_dir(PATHS['data_dir']),
    'protected_files_hashed': len(PROTECTED_AFTER), 'protected_unchanged': not CHANGED, 'protected_changed': CHANGED,
    'protected_sha256_before': PROTECTED_BEFORE, 'protected_sha256_after_equal_before': not CHANGED,
    'caveats': CAVEATS,
    'not_done': ['no training, model metrics, fitted bundles, probabilities, thresholds or coloured warnings', 'no zero-filled time panel (sizes estimated only)',
                 'no resolution selected; seen 2024/25 outcomes not used', 'no manual review performed yet (sample pending)'],
    'run_commands': run_commands(),
}
atomic_write_json(PATHS['report_dir'] / 'linkage_audit_manifest.json', MANIFEST)
print(STATUS, '| report:', PATHS['report_dir'], '| data (ignored):', PATHS['data_dir'])
if STATUS != 'complete':
    raise LinkageError('acceptance checks failed: ' + '; '.join(c['check'] for c in CHECKS if c['status'] != 'pass'))

## ผลลัพธ์

- `reports/v3/linkage_audit/`: ตารางรวม (csv), รูป 3 รูป, `linkage_audit_report_th.md`, `next_stage_protocol_proposal.md`, `linkage_audit_manifest.json`
- `data/v3/linkage_audit/` (gitignored): `event_linkage_audit.csv.gz`, `segment_inventory.csv.gz`, `segments_<R>m.geojson`, `road_pieces_audit.csv`, `piece_overlaps.csv`, `review_sample.csv`

คอลัมน์ chosen_* และ segment_id_* ใช้ได้เฉพาะแถวที่ `status_<tolerance>` = matched ของ tolerance ที่ใช้ (primary 50 ม.) ค่าว่าง = ไม่ทราบ ไม่ใช่ศูนย์